# Customer Health History (Rolling 90-Day Snapshot)

Daily **full snapshot** of `customer_health_profile` — captures all 34 columns + run_date (35 total) for maximum analytical flexibility. Maintains a 90-day rolling window for trend analysis, churn detection, and band transition tracking. Runs as a separate task after 3_15 completes the MERGE.

In [0]:
%sql
-- Daily health snapshot: append today's COMPLETE profile to a rolling 90-day history table.
-- Captures EVERY column from customer_health_profile (34 columns + run_date = 35 total).
-- One row per lead per day. Idempotent: deletes today's rows first (safe for re-runs).

CREATE TABLE IF NOT EXISTS crm_ingestion.gold.customer_health_history (
    run_date DATE,
    -- Identity
    lead_id STRING,
    status_label STRING,
    lead_name STRING,
    lead_display_name STRING,
    customer_emails ARRAY<STRING>,
    -- CSM
    CSM STRING,
    csm_name STRING,
    csm_email STRING,
    -- Commercial
    Contracted_Value STRING,
    Tier STRING,
    Avatar STRING,
    date_of_sale STRING,
    program STRING,
    setter_name STRING,
    closer_name STRING,
    sale_contract_value STRING,
    -- Engagement metrics
    calc_days_since_last_email INT,
    calc_days_since_last_meeting INT,
    missed_calls_14_days BIGINT,
    calc_days_until_next_meeting INT,
    calc_days_since_last_login INT,
    calc_days_since_last_meeting_calendly INT,
    sentiment STRING,
    days_since_last_message_from_client BIGINT,
    days_since_last_message_from_team_member BIGINT,
    -- Flags
    slack_engaged_flag INT,
    platform_engaged_flag INT,
    meeting_engaged_flag INT,
    engagement_pattern STRING,
    -- Scoring
    base_score INT,
    missed_call_adjustment INT,
    sentiment_adjustment INT,
    final_health_score INT,
    health_band STRING
);

DELETE FROM crm_ingestion.gold.customer_health_history WHERE run_date = CURRENT_DATE;

INSERT INTO crm_ingestion.gold.customer_health_history
SELECT
    CURRENT_DATE AS run_date,
    lead_id, status_label, lead_name, lead_display_name, customer_emails,
    CSM, csm_name, csm_email,
    Contracted_Value, Tier, Avatar, date_of_sale, program,
    setter_name, closer_name, sale_contract_value,
    calc_days_since_last_email, calc_days_since_last_meeting,
    missed_calls_14_days, calc_days_until_next_meeting,
    calc_days_since_last_login, calc_days_since_last_meeting_calendly,
    sentiment,
    days_since_last_message_from_client, days_since_last_message_from_team_member,
    slack_engaged_flag, platform_engaged_flag, meeting_engaged_flag,
    engagement_pattern,
    base_score, missed_call_adjustment, sentiment_adjustment,
    final_health_score, health_band
FROM crm_ingestion.gold.customer_health_profile

In [0]:
%sql
-- Rolling 90-day retention: delete snapshots older than 90 days
DELETE FROM crm_ingestion.gold.customer_health_history
WHERE run_date < DATE_SUB(CURRENT_DATE, 90)

In [0]:
%sql
-- Summary: client count and average score by run_date and health_band
SELECT
    run_date,
    health_band,
    COUNT(*) AS client_count,
    ROUND(AVG(final_health_score), 1) AS avg_score
FROM crm_ingestion.gold.customer_health_history
GROUP BY run_date, health_band
ORDER BY run_date DESC, health_band

In [0]:
%sql
select * from crm_ingestion.gold.customer_health_history